# CDS527 Group Project — Task 1: Text Classification System

**Dataset**: SMILE Twitter Emotion Dataset  
**Tool**: PySpark (primary)  
**Experiment protocol**: stratified 80/20 split, seed=42, primary metric = macro-F1

---

## Notebook Structure

| Section | Content | Status |
|---------|--------|--------|
| 0 | Environment Setup | ✅ |
| 1 | Data Loading & Schema | ✅ |
| 2 | EDA & Data Quality | ✅ |
| 3 | Preprocessing Pipeline | ✅ |
| 4 | Train/Test Split (fixed protocol) | ✅ |
| 5 | Section 1 — Baseline: TF-IDF + LR | ✅ |
| 6 | Section 2 — Model Comparison | ✅ |
| 7 | Section 3 — Representation Comparison | ✅ |
| 8 | Section 4 — Improvement & Analysis | ✅ |
| 9 | Final Results Summary | ✅ |

---

## Final Results

| Section | Best Configuration | macro-F1 |
|---------|-------------------|---------|
| S1 Baseline | TF-IDF unigram + LR | 0.2337 |
| S2 Model Comparison | TF-IDF unigram + CNB | 0.3332 |
| S3 Repr Comparison | All BOW+LR = 0.2337 (no gain) | 0.2337 |
| **S4 Improvement** | **LR + Class Weights (rp=0.5)** | **0.3453** |

---
## Section 0 — Environment Setup

In [ ]:
import os, time
# PySpark 3.1.2 requires Java 11; system default (Java 17) causes InaccessibleObjectException
os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/amazon-corretto-11.jdk/Contents/Home'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, trim, lower
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, HashingTF, Word2Vec, NGram
from pyspark.ml.classification import (
    LogisticRegression, NaiveBayes,
    DecisionTreeClassifier, RandomForestClassifier,
    OneVsRest, LinearSVC, MultilayerPerceptronClassifier
)
from pyspark.ml import Pipeline
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

SEED      = 42
DATA_PATH = '../smile-annotations-final.csv'
OUT_DIR   = '../输出'
LABEL_MAP = {0: 'surprise', 1: 'angry', 2: 'disgust', 3: 'happy', 4: 'sad'}
COLORS    = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CDS527-Task1') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)
print('Environment ready.')

---
## Section 1 — Data Loading & Schema

> **⚠️ Key note**: `multiLine=True` is mandatory. Some tweets contain embedded newlines. Without it, PySpark splits those rows and produces 1355 rows with 97 nulls instead of 1299 clean rows.

In [ ]:
df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=True,
                        multiLine=True, quote='"', escape='"')
df_raw = df_raw.dropDuplicates(['text'])   # 1 pair of text-identical rows (different tweetid, same label=happy)
df_raw = df_raw.withColumnRenamed('emotions', 'label')

print(f'Records after dedup: {df_raw.count()}')
df_raw.printSchema()
df_raw.show(5, truncate=80)

---
## Section 2 — EDA & Data Quality

In [ ]:
# 2.1 Label distribution
lc = df_raw.groupBy('label').count().orderBy('label').toPandas()
lc['emotion'] = lc['label'].apply(lambda x: LABEL_MAP.get(int(x), str(x)))
lc['pct']     = lc['count'] / lc['count'].sum() * 100
print('Label Distribution:')
print(lc[['label','emotion','count','pct']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order = [LABEL_MAP[i] for i in range(5)]
counts_ordered = lc.set_index('emotion').loc[order, 'count']
axes[0].bar(order, counts_ordered, color=COLORS, edgecolor='white')
axes[0].set_title('Label Distribution (count)', fontsize=13, fontweight='bold')
for i, (c, p) in enumerate(zip(counts_ordered, lc.set_index('emotion').loc[order,'pct'])):
    axes[0].text(i, c+8, f'{c}\n({p:.1f}%)', ha='center', fontsize=8.5)
axes[0].set_ylim(0, 1350)
axes[1].pie(counts_ordered, labels=order, autopct='%1.1f%%', colors=COLORS, startangle=140)
axes[1].set_title('Label Distribution (proportion)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig1_label_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.2 Text length distribution
df_pd = df_raw.toPandas()
df_pd['word_count'] = df_pd['text'].str.split().str.len()
df_pd['emotion']    = df_pd['label'].apply(lambda x: LABEL_MAP.get(int(x), str(x)))
print('Word count stats (all):')
print(df_pd['word_count'].describe().round(2))
print('\nMean word count per class:')
print(df_pd.groupby('emotion')['word_count'].mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_pd['word_count'], bins=25, color='steelblue', edgecolor='white')
axes[0].axvline(df_pd['word_count'].mean(),   color='red',    linestyle='--', label=f"mean={df_pd['word_count'].mean():.1f}")
axes[0].axvline(df_pd['word_count'].median(), color='orange', linestyle='--', label=f"median={df_pd['word_count'].median():.0f}")
axes[0].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
for i, emo in enumerate(order):
    grp = df_pd[df_pd['emotion']==emo]['word_count']
    axes[1].hist(grp, bins=20, alpha=0.55, color=COLORS[i], label=emo)
axes[1].set_title('Word Count by Emotion', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig2_word_count.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.3 Word clouds
import re
try:
    from wordcloud import WordCloud
    def make_wc_text(texts):
        t = ' '.join(texts)
        t = re.sub(r'https?://\S+|@\w+|&amp;|#', ' ', t)
        return re.sub(r'[^a-zA-Z\s]', ' ', t).lower()
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, emo in enumerate(order):
        texts = df_pd[df_pd['emotion']==emo]['text'].tolist()
        wc = WordCloud(width=400, height=280, background_color='white',
                       max_words=60, colormap='Blues' if emo=='happy' else 'Reds'
                       ).generate(make_wc_text(texts))
        axes[i].imshow(wc, interpolation='bilinear'); axes[i].axis('off')
        axes[i].set_title(f'{emo}  (n={len(texts)})', fontsize=12, fontweight='bold')
    axes[5].axis('off')
    plt.suptitle('Word Clouds by Emotion', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/fig5_wordclouds.png', bbox_inches='tight')
    plt.show()
except ImportError:
    print('wordcloud not installed, skipping.')

---
## Section 3 — Preprocessing Pipeline

Cleaning steps applied before PySpark tokenisation:

| Step | Operation | Reason |
|------|-----------|--------|
| 1 | Remove URLs | No semantic value for emotion |
| 2 | Remove @mentions | Twitter handles not informative |
| 3 | `&amp;` → `and` | HTML entity normalisation |
| 4 | Remove `#` but keep word | Preserve hashtag content |
| 5 | Keep alphabetic only | Remove punctuation, digits, emoji |
| 6 | Lowercase + trim | Normalise case |

In [ ]:
def clean_text(df, col_in='text', col_out='cleaned'):
    df = df.withColumn(col_out, regexp_replace(col(col_in),  r'http[s]?://\S+', ' '))
    df = df.withColumn(col_out, regexp_replace(col(col_out), r'@\w+', ' '))
    df = df.withColumn(col_out, regexp_replace(col(col_out), r'&amp;', 'and'))
    df = df.withColumn(col_out, regexp_replace(col(col_out), r'#(\w+)', r'\1'))
    df = df.withColumn(col_out, regexp_replace(col(col_out), r'[^a-zA-Z\s]', ' '))
    df = df.withColumn(col_out, lower(col(col_out)))
    df = df.withColumn(col_out, trim(regexp_replace(col(col_out), r'\s+', ' ')))
    return df

df = clean_text(df_raw)
df.select('text', 'cleaned', 'label').show(5, truncate=70)

---
## Section 4 — Train/Test Split

**Fixed protocol — do not change across any experiment**

| Parameter | Value |
|-----------|-------|
| Method | Stratified by `label` |
| Train fraction | 0.8 (≈80%) |
| Test fraction | remainder (≈20%) |
| Random seed | **42** |
| Effective samples | 1298 (after dedup) |

In [ ]:
label_vals = [r['label'] for r in df.select('label').distinct().collect()]
fractions  = {v: 0.8 for v in label_vals}
train = df.sampleBy('label', fractions=fractions, seed=SEED)
test  = df.subtract(train)
train.cache()
test.cache()

print(f'Train: {train.count()} rows  |  Test: {test.count()} rows')
print('\nTest label distribution:')
test.groupBy('label').count().orderBy('label').show()

---
## Evaluation Helper

Shared across all experiments.  
Primary metric: **macro-F1** (equal weight per class — appropriate for severe class imbalance).  
Also reports: weighted-F1, accuracy, per-class precision/recall/F1.

In [ ]:
def evaluate(predictions, experiment_name=''):
    preds_pd = predictions.select('label', 'prediction').toPandas()
    y_true = preds_pd['label'].astype(int)
    y_pred = preds_pd['prediction'].astype(int)
    macro_f1    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    accuracy    = (y_true == y_pred).mean()
    report = classification_report(y_true, y_pred,
                                   target_names=[LABEL_MAP[i] for i in sorted(LABEL_MAP)],
                                   zero_division=0)
    if experiment_name:
        print(f'\n=== {experiment_name} ===')
    print(f'  macro-F1    : {macro_f1:.4f}  ← primary metric')
    print(f'  weighted-F1 : {weighted_f1:.4f}')
    print(f'  accuracy    : {accuracy:.4f}')
    print('\nPer-class:')
    print(report)
    return {'macro_f1': macro_f1, 'weighted_f1': weighted_f1, 'accuracy': accuracy}

---
## Section 5 — Section 1: Baseline

**Pipeline**: `clean_text → RegexTokenizer → StopWordsRemover → CountVectorizer → IDF → LogisticRegression`  
**Feature**: TF-IDF (unigram, vocabSize=10000, minDF=2)  
**Model**: Logistic Regression (multinomial, maxIter=100, regParam=0.1)

In [ ]:
def make_tfidf_stages():
    """Shared TF-IDF (unigram) feature stages — used in Section 1 & 2."""
    tok = RegexTokenizer(inputCol='cleaned', outputCol='tokens', pattern='\\W', minTokenLength=2)
    rem = StopWordsRemover(inputCol='tokens', outputCol='filtered')
    cv  = CountVectorizer(inputCol='filtered', outputCol='raw_features', vocabSize=10000, minDF=2.0)
    idf = IDF(inputCol='raw_features', outputCol='features', minDocFreq=2)
    return [tok, rem, cv, idf]

lr_baseline = LogisticRegression(featuresCol='features', labelCol='label',
                                 maxIter=100, regParam=0.1, family='multinomial')
pipeline_s1 = Pipeline(stages=make_tfidf_stages() + [lr_baseline])
model_s1    = pipeline_s1.fit(train)
preds_s1    = model_s1.transform(test)

res_s1 = evaluate(preds_s1, 'Section 1 — Baseline: TF-IDF (unigram) + LR')
# Baseline result: macro-F1=0.2337, weighted-F1=0.8590, accuracy=0.8985

---
## Section 6 — Section 2: Model Comparison

**Fixed feature**: TF-IDF unigram (identical to baseline)  
**Variable**: classifier

| # | Model | PySpark Class | Notes |
|---|-------|--------------|-------|
| 1 | Logistic Regression | `LogisticRegression(family='multinomial')` | Baseline reference |
| 2 | Complement NaiveBayes | `NaiveBayes(modelType='complement')` | Designed for imbalanced text classification |
| 3 | Decision Tree | `DecisionTreeClassifier(maxDepth=10)` | Interpretable, handles multi-class natively |
| 4 | Random Forest | `RandomForestClassifier(numTrees=100)` | Ensemble; may overfit to majority class |
| 5 | OneVsRest + LinearSVC | `OneVsRest(classifier=LinearSVC())` | Trains 5 binary classifiers |

> Cache is released after each model to avoid memory pressure.

In [ ]:
import time

s2_models = [
    ('LR',      LogisticRegression(featuresCol='features', labelCol='label',
                    maxIter=100, regParam=0.1, family='multinomial'),
     'Logistic Regression'),
    ('CNB',     NaiveBayes(featuresCol='features', labelCol='label', modelType='complement'),
     'Complement NaiveBayes'),
    ('DT',      DecisionTreeClassifier(featuresCol='features', labelCol='label',
                    seed=SEED, maxDepth=10),
     'Decision Tree'),
    ('RF',      RandomForestClassifier(featuresCol='features', labelCol='label',
                    seed=SEED, numTrees=100),
     'Random Forest'),
    ('OVR-SVC', OneVsRest(classifier=LinearSVC(featuresCol='features', labelCol='label',
                    maxIter=100), featuresCol='features', labelCol='label'),
     'OneVsRest + LinearSVC'),
]

s2_results = []
for short, classifier, display in s2_models:
    print(f'\n>>> {display} ...')
    t0 = time.time()
    pipeline = Pipeline(stages=make_tfidf_stages() + [classifier])
    model    = pipeline.fit(train)
    preds    = model.transform(test).cache()
    elapsed  = time.time() - t0

    res = evaluate(preds, f'{display} ({elapsed:.1f}s)')
    s2_results.append({'model': display, 'short': short,
                       'macro_f1': res['macro_f1'],
                       'weighted_f1': res['weighted_f1'],
                       'accuracy': res['accuracy'],
                       'train_sec': round(elapsed, 1)})
    # Release this model's prediction cache before next iteration
    preds.unpersist()
    del model, pipeline, preds

In [ ]:
# Section 2 summary table
df_s2 = pd.DataFrame(s2_results)[['model','macro_f1','weighted_f1','accuracy','train_sec']]
df_s2 = df_s2.sort_values('macro_f1', ascending=False).reset_index(drop=True)
df_s2.columns = ['Model', 'macro-F1 ↑', 'weighted-F1', 'accuracy', 'train(s)']
print('\n=== Section 2: Model Comparison Summary (sorted by macro-F1) ===')
print(df_s2.to_string(index=False))

In [ ]:
# Section 2 visualisation
COLORS_5 = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels_x  = [r['model'] for r in s2_results]
mf1       = [r['macro_f1'] for r in s2_results]
wf1       = [r['weighted_f1'] for r in s2_results]
acc       = [r['accuracy'] for r in s2_results]

bars = axes[0].bar(labels_x, mf1, color=COLORS_5, edgecolor='white', width=0.55)
axes[0].set_title('Section 2 — macro-F1 by Model\n(primary metric)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('macro-F1')
axes[0].set_ylim(0, max(mf1)*1.25)
axes[0].tick_params(axis='x', labelsize=8.5)
for bar, v in zip(bars, mf1):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.004, f'{v:.4f}',
                 ha='center', fontsize=9, fontweight='bold')

x = np.arange(len(s2_results)); w = 0.25
axes[1].bar(x-w, mf1, width=w, label='macro-F1',    color='#3498db', alpha=0.85)
axes[1].bar(x,   wf1, width=w, label='weighted-F1', color='#e74c3c', alpha=0.85)
axes[1].bar(x+w, acc, width=w, label='accuracy',    color='#2ecc71', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels_x, fontsize=8.5)
axes[1].set_title('Section 2 — All Metrics', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.15); axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig6_s2_model_comparison.png', bbox_inches='tight')
plt.show()
print('Saved fig6_s2_model_comparison.png')

### Section 2 — Key Findings

| Model | macro-F1 | Interpretation |
|-------|---------|----------------|
| **Complement NaiveBayes** | **0.3332** | Best — recognises angry (F1=0.55), surprise (0.09), sad (0.12). Designed for imbalanced text. |
| OneVsRest + LinearSVC | 0.3142 | Second best — surprise (0.22), angry (0.40). Slow (130s). |
| Decision Tree | 0.2947 | angry (0.52), others 0. Fast (3s). |
| Logistic Regression | 0.2337 | Baseline — only angry partially (0.22). |
| Random Forest | 0.1889 | **Worst** — all minority classes 0 despite accuracy=89%. Overwhelmed by class imbalance. |

**Why accuracy misleads**: RF has 89.5% accuracy but worst macro-F1. It predicts almost all samples as `happy`.  
**CNB chosen for Section 3**: Best macro-F1, fast (1s), handles class imbalance by design.

---
## Section 7 — Section 3: Representation Comparison

**Fixed classifier**: Logistic Regression (same params as baseline)  
**Variable**: feature representation

| ID | Representation | Notes |
|----|---------------|-------|
| R1 | TF-IDF (unigram) | Baseline reference |
| R2 | TF-IDF (1,2-gram) | Unigrams + bigrams merged via VectorAssembler |
| R3 | CountVectorizer (no IDF) | Raw term frequency, no IDF weighting |
| R4 | Word2Vec (dim=100) | Average word embeddings, dense representation |

> **Key finding (within current fixed-LR protocol)**: Under this setup, R1, R2, R3 produce identical results (macro-F1=0.2337, per-class values equal).  
> With 89% class imbalance and regParam=0.1, the gradient signal from the majority class dominates LR's optimisation.  
> In this configuration, changes within the BOW feature family did not produce distinguishable performance differences.  
> This suggests that class imbalance handling (e.g., class weighting) may be a more impactful lever than feature choice — to be tested in Section 4.  
> Cache is released after each method.

In [ ]:
from pyspark.ml.feature import VectorAssembler

# R1: TF-IDF unigram (reference)
def make_pipe_r1():
    tok = RegexTokenizer(inputCol='cleaned', outputCol='tokens', pattern='\\W', minTokenLength=2)
    rem = StopWordsRemover(inputCol='tokens', outputCol='filtered')
    cv  = CountVectorizer(inputCol='filtered', outputCol='raw_features', vocabSize=10000, minDF=2.0)
    idf = IDF(inputCol='raw_features', outputCol='features', minDocFreq=2)
    lr  = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.1, family='multinomial')
    return Pipeline(stages=[tok, rem, cv, idf, lr])

# R2: TF-IDF 1,2-gram — unigrams + bigrams, VectorAssembler
def make_pipe_r2():
    tok     = RegexTokenizer(inputCol='cleaned', outputCol='tokens', pattern='\\W', minTokenLength=2)
    rem     = StopWordsRemover(inputCol='tokens', outputCol='filtered')
    ng      = NGram(n=2, inputCol='filtered', outputCol='bigrams')
    cv_uni  = CountVectorizer(inputCol='filtered', outputCol='feat_uni', vocabSize=8000, minDF=2.0)
    cv_bi   = CountVectorizer(inputCol='bigrams',  outputCol='feat_bi',  vocabSize=5000, minDF=2.0)
    idf_uni = IDF(inputCol='feat_uni', outputCol='idf_uni', minDocFreq=2)
    idf_bi  = IDF(inputCol='feat_bi',  outputCol='idf_bi',  minDocFreq=2)
    asm     = VectorAssembler(inputCols=['idf_uni','idf_bi'], outputCol='features')
    lr      = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.1, family='multinomial')
    return Pipeline(stages=[tok, rem, ng, cv_uni, cv_bi, idf_uni, idf_bi, asm, lr])

# R3: CountVectorizer only (no IDF)
def make_pipe_r3():
    tok = RegexTokenizer(inputCol='cleaned', outputCol='tokens', pattern='\\W', minTokenLength=2)
    rem = StopWordsRemover(inputCol='tokens', outputCol='filtered')
    cv  = CountVectorizer(inputCol='filtered', outputCol='features', vocabSize=10000, minDF=2.0)
    lr  = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.1, family='multinomial')
    return Pipeline(stages=[tok, rem, cv, lr])

# R4: Word2Vec (average word embeddings, dim=100)
def make_pipe_r4():
    tok = RegexTokenizer(inputCol='cleaned', outputCol='tokens', pattern='\\W', minTokenLength=2)
    rem = StopWordsRemover(inputCol='tokens', outputCol='filtered')
    w2v = Word2Vec(vectorSize=100, minCount=1, inputCol='filtered', outputCol='features', seed=SEED)
    lr  = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.1, family='multinomial')
    return Pipeline(stages=[tok, rem, w2v, lr])

s3_configs = [
    ('R1', 'TF-IDF (unigram)',         make_pipe_r1),
    ('R2', 'TF-IDF (1,2-gram)',        make_pipe_r2),
    ('R3', 'CountVectorizer (no IDF)', make_pipe_r3),
    ('R4', 'Word2Vec (dim=100)',        make_pipe_r4),
]

s3_results = []
for short, display, make_pipeline in s3_configs:
    print(f'\n>>> {display} ...')
    t0    = time.time()
    pipe  = make_pipeline()
    model = pipe.fit(train)
    preds = model.transform(test).cache()
    t1    = time.time()
    res   = evaluate(preds, f'{display} ({t1-t0:.1f}s)')
    s3_results.append({'repr': display, 'short': short,
                       'macro_f1': res['macro_f1'],
                       'weighted_f1': res['weighted_f1'],
                       'accuracy': res['accuracy'],
                       'train_sec': round(t1-t0, 1)})
    preds.unpersist()          # release cache after each method
    del model, pipe, preds

### Section 3 — Key Finding: Feature Representation (Within Current Fixed-LR Setup)

**In this fixed-LR setting, R1/R2/R3 produce identical results.** Adding bigrams or removing IDF did not change outcomes with LR under 89% class imbalance.

**Why**: With 89% `happy` samples, LR's cross-entropy loss is dominated by the majority class gradient. Under regParam=0.1 and no class weighting, the unigram, bigram, and raw-count feature spaces all fail to provide sufficient discriminative signal for the minority classes, leading to similar prediction distributions. Note this behaviour may differ with a different classifier (e.g., CNB) or after class-weight adjustment.

**Word2Vec (R4)** is worse: averaged dense embeddings compress the discriminative rare-word signals that sparse BOW methods partially retain. With ~30–50 minority-class training samples, the embedding space is overwhelmingly shaped by `happy` context words.

| Representation | macro-F1 | vs Baseline |
|----------------|---------|-------------|
| TF-IDF unigram (R1) | 0.2337 | — |
| TF-IDF 1,2-gram (R2) | 0.2337 | 0.0000 |
| CountVectorizer (R3) | 0.2337 | 0.0000 |
| Word2Vec (R4) | 0.1889 | **−0.0448** |

**Interim conclusion (current protocol)**: Within the fixed-LR, no-class-weighting setup, changes in the BOW feature family did not alter model behaviour. Class imbalance handling is the higher-priority direction to explore in Section 4 — this does not preclude a contribution from feature choice under different classifier settings.

In [ ]:
# Section 3 summary + visualisation
df_s3 = pd.DataFrame(s3_results)[['repr','macro_f1','weighted_f1','accuracy','train_sec']]
df_s3.columns = ['Representation', 'macro-F1', 'weighted-F1', 'accuracy', 'train(s)']
print('\n=== Section 3: Representation Comparison Summary ===')
print(df_s3.to_string(index=False))
# Key: R1=R2=R3 identical; R4 worse

COLORS_4 = ['#3498db','#e74c3c','#2ecc71','#f39c12']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels_x = [r['repr'] for r in s3_results]
mf1_s3   = [r['macro_f1'] for r in s3_results]

bars = axes[0].bar(labels_x, mf1_s3, color=COLORS_4, edgecolor='white', width=0.55)
axes[0].axhline(0.2337, color='gray', linestyle='--', linewidth=1.2, label='S1 Baseline')
axes[0].set_title('Section 3 — Representation Comparison\nmacro-F1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('macro-F1'); axes[0].set_ylim(0, 0.35)
axes[0].tick_params(axis='x', labelsize=8.5); axes[0].legend(fontsize=9)
for bar, v in zip(bars, mf1_s3):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.4f}',
                 ha='center', fontsize=9, fontweight='bold')

x = range(len(s3_results)); w = 0.25
axes[1].bar([i-w for i in x], mf1_s3,
            width=w, label='macro-F1',    color='#3498db', alpha=0.85)
axes[1].bar([i   for i in x], [r['weighted_f1'] for r in s3_results],
            width=w, label='weighted-F1', color='#e74c3c', alpha=0.85)
axes[1].bar([i+w for i in x], [r['accuracy'] for r in s3_results],
            width=w, label='accuracy',    color='#2ecc71', alpha=0.85)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(labels_x, fontsize=8.5)
axes[1].set_title('Section 3 — All Metrics', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.15); axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/fig7_s3_repr_comparison.png', bbox_inches='tight')
plt.show()

---
## Section 8 — Section 4: Improvement & Deep Analysis

### Part A: Class Weighting + Hyperparameter Tuning

**Motivation from Section 3**: In the fixed-LR, no-weighting protocol, changes within the BOW feature family did not produce distinguishable results. This points to class imbalance handling as the higher-priority lever.

**Strategy**: Assign inverse-frequency class weights (`weightCol`) to up-weight minority classes in LR's cross-entropy loss.

| Approach | Description |
|----------|-------------|
| LR + class weights | weight[c] = n_samples / (n_classes × count[c]) — multiplies minority loss by 5–21× |
| regParam sweep | [0.01, 0.05, 0.1, 0.5, 1.0] with class weights — find optimal regularisation |
| CNB (reference) | Best from Section 2 (macro-F1=0.3332), re-run for direct comparison |

### Part B: Deep Analysis

- Confusion matrix (counts + row-normalised) for best model
- Per-class precision / recall / F1 bar chart
- Hardest misclassification pairs
- Training time vs performance scatter across all experiments

In [ ]:
# ── Section 4 Part A: Compute inverse-frequency class weights ────────────────
from pyspark.sql.functions import when as sf_when, col as sf_col

label_counts_pd = train.groupBy('label').count().orderBy('label').toPandas()
n_samp = int(label_counts_pd['count'].sum())
n_cls  = len(label_counts_pd)

weight_dict = {}
for _, row in label_counts_pd.iterrows():
    lbl = int(row['label'])
    weight_dict[lbl] = n_samp / (n_cls * int(row['count']))

print('Class weights (inverse-frequency):')
for lbl in sorted(weight_dict):
    print(f'  label={lbl}  {LABEL_MAP[lbl]:10s}  weight={weight_dict[lbl]:.3f}')

# Add weight column to train (lazy — uses cached train)
w_expr = sf_when(sf_col('label') == 0, weight_dict[0])
for lbl in range(1, n_cls):
    w_expr = w_expr.when(sf_col('label') == lbl, weight_dict[lbl])
w_expr = w_expr.otherwise(1.0)
train_w = train.withColumn('class_weight', w_expr)

In [ ]:
# ── Section 4 Part A: Run experiments ────────────────────────────────────────
s4a_results   = []
best_s4_mf1   = 0.0
best_s4_preds = None
best_s4_label = ''

# LR + class weights: regParam sweep
regparam_values = [0.01, 0.05, 0.1, 0.5, 1.0]
for rp in regparam_values:
    display = f'LR+Weight (rp={rp})'
    print(f'\n>>> {display} ...')
    t0   = time.time()
    lr   = LogisticRegression(featuresCol='features', labelCol='label',
                              weightCol='class_weight',
                              maxIter=200, regParam=rp, family='multinomial')
    pipe  = Pipeline(stages=make_tfidf_stages() + [lr])
    model = pipe.fit(train_w)
    preds = model.transform(test).cache()
    t1    = time.time()
    res   = evaluate(preds, f'{display} ({t1-t0:.1f}s)')
    s4a_results.append({'display': display, 'type': 'LR+Weight', 'regParam': rp,
                        'macro_f1': res['macro_f1'], 'weighted_f1': res['weighted_f1'],
                        'accuracy': res['accuracy'], 'train_sec': round(t1-t0, 1)})
    if res['macro_f1'] > best_s4_mf1:
        if best_s4_preds is not None:
            best_s4_preds.unpersist()
        best_s4_mf1   = res['macro_f1']
        best_s4_preds = preds
        best_s4_label = display
    else:
        preds.unpersist()
    del model, pipe

# CNB (Section 2 best, reference)
print('\n>>> CNB (S2 reference) ...')
t0   = time.time()
cnb  = NaiveBayes(featuresCol='features', labelCol='label', modelType='complement')
pipe = Pipeline(stages=make_tfidf_stages() + [cnb])
model = pipe.fit(train)
preds = model.transform(test).cache()
t1    = time.time()
res   = evaluate(preds, f'CNB S2 reference ({t1-t0:.1f}s)')
s4a_results.append({'display': 'CNB (S2 reference)', 'type': 'CNB', 'regParam': None,
                    'macro_f1': res['macro_f1'], 'weighted_f1': res['weighted_f1'],
                    'accuracy': res['accuracy'], 'train_sec': round(t1-t0, 1)})
if res['macro_f1'] > best_s4_mf1:
    if best_s4_preds is not None:
        best_s4_preds.unpersist()
    best_s4_mf1   = res['macro_f1']
    best_s4_preds = preds
    best_s4_label = 'CNB (S2 reference)'
else:
    preds.unpersist()
del model, pipe

print(f'\n✓ Best S4 model: {best_s4_label}  macro-F1={best_s4_mf1:.4f}')

In [ ]:
# ── Section 4 Part A: Summary table + visualisation ──────────────────────────
df_s4a = pd.DataFrame(s4a_results)
df_s4a_sorted = df_s4a.sort_values('macro_f1', ascending=False).reset_index(drop=True)
print('\n=== Section 4A: Improvement Results (sorted by macro-F1) ===')
print(df_s4a_sorted[['display','macro_f1','weighted_f1','accuracy','train_sec']].to_string(index=False))
print('\n  Reference: S1 Baseline (LR, no weight) macro-F1=0.2337')
print(  '  Reference: S2 Best     (CNB, default)  macro-F1=0.3332')

COLORS_S4 = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']
r_sorted  = sorted(s4a_results, key=lambda x: x['macro_f1'], reverse=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(range(len(r_sorted)), [r['macro_f1'] for r in r_sorted],
                   color=COLORS_S4[:len(r_sorted)], edgecolor='white', width=0.6)
axes[0].axhline(0.2337, color='gray',   linestyle='--', lw=1.2, label='S1 Baseline (LR)')
axes[0].axhline(0.3332, color='orange', linestyle='--', lw=1.2, label='S2 Best (CNB)')
axes[0].set_xticks(range(len(r_sorted)))
axes[0].set_xticklabels([r['display'] for r in r_sorted], rotation=30, ha='right', fontsize=8)
axes[0].set_title('Section 4A — Improvement Results\nmacro-F1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('macro-F1')
axes[0].set_ylim(0, max(r['macro_f1'] for r in r_sorted) * 1.28)
axes[0].legend(fontsize=8)
for bar, r in zip(bars, r_sorted):
    axes[0].text(bar.get_x()+bar.get_width()/2, r['macro_f1']+0.005,
                 f'{r["macro_f1"]:.4f}', ha='center', fontsize=8, fontweight='bold')

lr_r = [r for r in s4a_results if r['type'] == 'LR+Weight']
axes[1].plot([r['regParam'] for r in lr_r], [r['macro_f1'] for r in lr_r],
             'o-', color='#3498db', lw=2, markersize=8, label='LR+ClassWeight')
axes[1].axhline(0.2337, color='gray',   linestyle='--', lw=1.2, label='S1 Baseline (LR)')
axes[1].axhline(0.3332, color='orange', linestyle='--', lw=1.2, label='S2 Best (CNB)')
axes[1].set_xscale('log'); axes[1].set_xlabel('regParam (log scale)')
axes[1].set_ylabel('macro-F1')
axes[1].set_title('LR + Class Weights: regParam Sensitivity', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8)
for r in lr_r:
    axes[1].annotate(f'{r["macro_f1"]:.4f}', (r['regParam'], r['macro_f1']),
                     textcoords='offset points', xytext=(0, 9), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/fig8_s4_improvement.png', bbox_inches='tight')
plt.show()
print('Saved fig8_s4_improvement.png')

In [ ]:
# ── Section 4 Part B: Deep Analysis ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

print(f'Deep analysis on: {best_s4_label}  (macro-F1={best_s4_mf1:.4f})')
preds_pd = best_s4_preds.select('label', 'prediction').toPandas()
y_true = preds_pd['label'].astype(int).values
y_pred = preds_pd['prediction'].astype(int).values
emotion_names = [LABEL_MAP[i] for i in range(5)]

# Confusion matrix (counts + row-normalised)
cm = confusion_matrix(y_true, y_pred, labels=list(range(5)))
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm  = np.where(row_sums > 0, cm.astype(float) / row_sums, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, mat, title, is_float in zip(
    axes,
    [cm, cm_norm],
    [f'Confusion Matrix (counts)\n{best_s4_label}',
     f'Confusion Matrix (row-normalised)\n{best_s4_label}'],
    [False, True]
):
    im = ax.imshow(mat, cmap='Blues', vmin=0)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(emotion_names, rotation=45, ha='right')
    ax.set_yticklabels(emotion_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)
    for i in range(5):
        for j in range(5):
            val = mat[i, j]
            txt = f'{val:.2f}' if is_float else str(int(val))
            ax.text(j, i, txt, ha='center', va='center',
                    color='white' if val > mat.max()*0.55 else 'black', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/fig9_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved fig9_confusion_matrix.png')

# Release best model predictions
best_s4_preds.unpersist()

In [ ]:
# ── Section 4 Part B: Per-class metrics + time vs performance ─────────────────
precision, recall, f1_cls, support = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(5)), zero_division=0)

print('Per-class metrics:')
print(f'{"Emotion":12s} {"Precision":>10s} {"Recall":>8s} {"F1":>8s} {"Support":>9s}')
for i in range(5):
    print(f'{LABEL_MAP[i]:12s} {precision[i]:10.4f} {recall[i]:8.4f} {f1_cls[i]:8.4f} {support[i]:9d}')

confused_pairs = sorted(
    [(cm[i,j], LABEL_MAP[i], LABEL_MAP[j])
     for i in range(5) for j in range(5) if i != j and cm[i,j] > 0],
    reverse=True)
print('\nHardest misclassification pairs:')
for cnt, tl, pl in confused_pairs[:10]:
    print(f'  True={tl:10s} -> Predicted={pl:10s}  count={cnt}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_idx = np.arange(5); w = 0.25
axes[0].bar(x_idx-w, precision, width=w, label='Precision', color='#3498db', alpha=0.85)
axes[0].bar(x_idx,   recall,    width=w, label='Recall',    color='#e74c3c', alpha=0.85)
axes[0].bar(x_idx+w, f1_cls,    width=w, label='F1',        color='#2ecc71', alpha=0.85)
axes[0].set_xticks(x_idx); axes[0].set_xticklabels(emotion_names, fontsize=10)
axes[0].set_ylim(0, 1.15); axes[0].set_ylabel('Score')
axes[0].set_title(f'Per-class Metrics\n{best_s4_label}', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
for i in range(5):
    for v, off in zip([precision[i], recall[i], f1_cls[i]], [-w, 0, w]):
        if v > 0.01:
            axes[0].text(i+off, v+0.02, f'{v:.2f}', ha='center', fontsize=7)

scatter_all = [
    ('S1 Baseline', 0.2337, 8.3,   '#95a5a6'),
    ('S2 CNB',      0.3332, 1.0,   '#95a5a6'),
    ('S2 OVR-SVC',  0.3142, 129.7, '#95a5a6'),
    ('S2 DT',       0.2947, 2.9,   '#95a5a6'),
    ('S2 RF',       0.1889, 5.4,   '#95a5a6'),
] + [(r['display'], r['macro_f1'], r['train_sec'], '#e74c3c') for r in s4a_results]
for lbl, mf1_v, sec, clr in scatter_all:
    axes[1].scatter(sec, mf1_v, s=70, color=clr, zorder=3, edgecolors='white', linewidths=0.5)
    axes[1].annotate(lbl, (sec, mf1_v), textcoords='offset points', xytext=(4, 3), fontsize=6.5)
axes[1].axhline(0.3332, color='orange', linestyle='--', lw=1, alpha=0.7, label='S2 Best (CNB)')
axes[1].set_xscale('log')
axes[1].set_xlabel('Training Time (s)'); axes[1].set_ylabel('macro-F1')
axes[1].set_title('Training Time vs Performance\n(all experiments)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/fig10_per_class_and_time.png', bbox_inches='tight')
plt.show()
print('Saved fig10_per_class_and_time.png')

### Section 4 — Key Findings

#### Part A: Class Weighting Results

| Configuration | macro-F1 | vs S1 Baseline | vs S2 Best (CNB) |
|---------------|---------|----------------|-----------------|
| **LR+Weight (rp=0.5)** | **0.3453** | **+0.1116 (+47.7%)** | **+0.0121 (+3.6%)** |
| LR+Weight (rp=0.05) | 0.3375 | +0.1038 | +0.0043 |
| LR+Weight (rp=1.0) | 0.3349 | +0.1012 | +0.0017 |
| CNB (S2 reference) | 0.3332 | +0.0995 | — |
| LR+Weight (rp=0.1) | 0.3331 | +0.0994 | −0.0001 |
| LR+Weight (rp=0.01) | 0.3313 | +0.0976 | −0.0019 |

**Insight**: Class weighting successfully breaks through the S3 ceiling. LR+Weight (rp=0.5) achieves the best macro-F1=0.3453, slightly surpassing CNB. The trade-off: accuracy drops from 90% to 80% as the model is now more willing to predict minority classes.

**regParam sensitivity**: Lower regParam (0.01) under-regularises and performs slightly worse; the sweet spot is rp=0.5 where regularisation is strong enough to stabilise the weighted loss.

#### Part B: Deep Analysis (best model: LR+Weight rp=0.5)

| Class | Precision | Recall | F1 | Support |
|-------|-----------|--------|----|---------|
| surprise | 0.09 | 0.33 | 0.14 | 6 |
| **angry** | **0.70** | **0.47** | **0.56** | 15 |
| disgust | 0.00 | 0.00 | 0.00 | 3 |
| happy | 0.94 | 0.86 | 0.90 | 238 |
| sad | 0.08 | 0.25 | 0.12 | 4 |

**Hardest pairs**: `happy → surprise` (18 cases), `happy → sad` (8 cases) — class weighting causes some happy samples to be mispredicted as minority classes. This is the expected precision–recall trade-off.

**disgust**: Still F1=0.00 across all models. With only 3 test samples and 10 training samples, this is a **data limitation**, not a modelling failure. Report should address this explicitly.

---
## Section 9 — Final Results Summary

**Dataset**: SMILE Twitter Emotion Dataset — 1,298 tweets from 13 British museum accounts, labelled into 5 emotion classes (happy: 89.1%, disgust: 1.0%). Severe class imbalance is the defining challenge throughout. All experiments use a fixed stratified 80/20 split (seed=42, train=1,032 / test=266), evaluated primarily by macro-F1 (equal weight per class, appropriate for imbalanced multi-class classification).

**Section 1 — Baseline**: TF-IDF unigram + Logistic Regression. macro-F1=0.2337, accuracy=0.8985. High accuracy is misleading: the model nearly always predicts *happy*. Only *angry* is partially recognised (F1=0.22); all other minority classes score zero.

**Section 2 — Model Comparison** (fixed TF-IDF features, 5 classifiers): Complement Naive Bayes achieves the best macro-F1=0.3332 (+42.5% over baseline), with angry F1=0.55. Random Forest, despite accuracy=89.5%, scores the lowest macro-F1=0.1889 with all minority classes at zero — confirming that accuracy is an unreliable metric under class imbalance.

**Section 3 — Representation Comparison** (fixed LR, 4 representations): TF-IDF unigram, bigram, and CountVectorizer all produce identical results (macro-F1=0.2337). Word2Vec performs worst (0.1889). Within this fixed-LR protocol, feature representation choice had no measurable effect, pointing to class imbalance handling as the more impactful variable.

**Section 4 — Improvement & Analysis**: Applying inverse-frequency class weights to LR (regParam=0.5) achieves macro-F1=0.3453 — the best result across all experiments (+47.7% over baseline, +3.6% over CNB). The accuracy–macro-F1 trade-off (accuracy drops to 80%) reflects the model actively predicting minority classes. *disgust* remains at F1=0.00 throughout: with only 10 training and 3 test samples, this is a dataset-level limitation.

**Key conclusions**: (1) macro-F1 is the only reliable metric in this setting; accuracy is dominated by the 89% majority class. (2) Class imbalance handling (inverse-frequency class weighting) is the most effective lever — outperforming feature engineering changes. (3) Best system: **LR + TF-IDF unigram + inverse-frequency class weights (regParam=0.5), macro-F1=0.3453**.

In [ ]:
# Section 9 is a text summary above — no additional code required.

In [ ]:
# Release train/test cache before stopping
train.unpersist()
test.unpersist()
spark.stop()
print('Spark session stopped.')

## Section 10 — Supplementary Experiments (Appended)

为避免改动原有 Section 1–4 的主实验流程，下面把补做实验统一追加到 Notebook 最底部。
这一部分不插入原有代码块，而是作为补充章节单独呈现，便于老师或组员区分“原主线实验”和“后补实验”。

本补充章节覆盖：
- Section 3 Part B：`GloVe + LR`、`BERT embedding + LR`
- Section 4：`Optional MLP + Word2Vec`
- Section 4：`ablation study`
- 总表与按 Section 整理后的输出入口


### 10.1 读取补做实验的输出目录

下面这段代码只负责定位 `输出/` 目录，并读取补做实验生成的 CSV、图表和汇总表。
这样做的目的是保持原 Notebook 主体不变，同时让补充实验结果可以在 Notebook 中直接展示。


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

NB_CWD = Path.cwd()
PROJECT_ROOT = NB_CWD.parent if NB_CWD.name == '工作区' else NB_CWD
OUTPUT_DIR = PROJECT_ROOT / '输出'
DATA_DIR = OUTPUT_DIR / 'data'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'

print('Project root:', PROJECT_ROOT)
print('Output dir  :', OUTPUT_DIR)


### 10.2 Section 3 Part B：GloVe 与 BERT embedding

这部分补做了原先未纳入主线的两种预训练表示方法。
下面的代码读取结构化结果表，并展示 Section 3 Part B 的对比图，方便直接比较 `macro-F1`、`weighted-F1` 和 `accuracy`。


In [ ]:
s3b_results = pd.read_csv(DATA_DIR / 'results_s3_partb_embeddings.csv').sort_values('macro_f1', ascending=False)
display(s3b_results)
display(Image(filename=str(FIG_DIR / 'fig11_s3b_embeddings.png')))


### 10.3 Section 4：Optional MLP + Word2Vec

这部分补做了官方路线图中列出的可选项 `MLP + Word2Vec`。
下面的代码读取该实验的结果表，用于确认它是否比现有主线方案更优。


In [ ]:
mlp_w2v_results = pd.read_csv(DATA_DIR / 'results_s4_optional_mlp_word2vec.csv')
display(mlp_w2v_results)


### 10.4 Section 4：Ablation Study

这部分补做了消融实验，用来回答“到底是哪一部分真正带来了提升”。
下面的代码读取 ablation 结果表，并展示对应图表。这个补充实验非常重要，因为它不仅补齐了官方路线图，还发现了新的最佳配置。


In [ ]:
ablation_results = pd.read_csv(DATA_DIR / 'results_s4_ablation.csv').sort_values('macro_f1', ascending=False)
display(ablation_results)
display(Image(filename=str(FIG_DIR / 'fig12_s4_ablation.png')))


### 10.5 全部实验总表与排名表

最后这段代码读取新的总表与排名表。
其中 `overall_experiment_metrics` 已按 section 汇总，并明确标记了 `baseline`；`macro_f1_ranking` 则按主指标从高到低排序，方便直接写入报告或 PPT。


In [ ]:
overall_metrics = pd.read_csv(TABLE_DIR / 'overall_experiment_metrics.csv')
macro_ranking = pd.read_csv(TABLE_DIR / 'macro_f1_ranking.csv')
roadmap_status = pd.read_csv(TABLE_DIR / 'official_roadmap_status.csv')

print('Overall experiment metrics:')
display(overall_metrics)
print('Macro-F1 ranking:')
display(macro_ranking)
print('Official roadmap status:')
display(roadmap_status)


### 10.6 补充结论

补做实验后的结论需要和原主线区分开表述：
- `BERT embedding + LR` 提高了 `accuracy`，但没有成为最终最优 `macro-F1` 配置
- `MLP + Word2Vec` 没有带来收益
- `ablation study` 反而发现了新的最优方案：`A4 Light cleaning only`
- 因此，如果最终报告以 `macro-F1` 为主指标，当前推荐的最优方案应更新为 `A4`
